# Project 3 (Group Project)

As a team, you will identify a binary target you want to predict, build a classification model to predict the target. You will try various machine learning models, conduct hyper-parameter tuning and cross-validation to find the best model. You will discuss/visualize your results and make a presentation at the last day of this class 6/26. In addition, you will create a digital poster to summarize your findings.

Please submit the completed notebook, slides, digitial poster on Canvas on the due date.

As an example, I will show you how to build a model to predict whether a member will likely to cancel a booking.

Below is some sample code to give you some ideas of running a classification model using pyspark.

For your convinence, I created a python file (helper_functions_ml.py) that includes a few functions to evalute the output of the classification models. Please put it in the same folder as your main notebook. 

# Import Note:

You don’t need to perform hyperparameter tuning on every model. Instead, run the base version of each algorithm, compare their performance, and then fine-tune the best-performing one.

In [ ]:
# append path to utilities to system path
import sys
#sys.path.append('/opt/shared/')

from pyspark.sql import SparkSession
import numpy as np
from wordcloud import WordCloud
from PIL import Image
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import udf, desc, col, isnull, to_timestamp, to_date, trim, year, month, when, sum, lit, ntile, rank
from pyspark.sql.types import StringType, IntegerType, DoubleType, ArrayType, MapType, LongType, BooleanType
from pyspark.sql.window import Window
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.functions import vector_to_array
import helper_functions_ml as M
import pandas as pd

import utilities as U
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')

# change the account name to your email account
account='sslachta'

# define a data path to store the data 
data_path='/net/clusterhn/home/'+account+'/MSDS630/aaa_data/'

# Create a Spark session

In [ ]:
# check if the Spark session is active. If it is activate, close it
try:
    if spark:
        spark.stop()
except:
    pass    

spark = (SparkSession.builder.appName("AAA Data Analytics Project")
        .config("spark.port.maxRetries", "100")
        .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")  # This configuration allow the duplicate keys in the map data type.
        .config("spark.driver.memory", "16g")
        .getOrCreate())

# configure the log level (defaulty is WWARN)
spark.sparkContext.setLogLevel('ERROR')

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")

# Import features

In [ ]:
#df=spark.read.parquet(data_path+'transformed_features_cruiser_cancellation_v2.parquet')
df=spark.read.parquet(data_path+'travel_book_summary_cleaned.parquet')

In [ ]:
df.columns

# Build a model to predict whether a members has booked a cruise.

In [ ]:
df.groupBy('travel_type_description').count().orderBy(desc('count')).show()

In [ ]:
travel_type = df.groupBy('individual_id',  'travel_type_description').count()

travel_type1 =travel_type.groupBy('individual_id').pivot('travel_type_description').agg(sum('count')).fillna(0)
travel_type2 = travel_type1.withColumn('Cruise', when(col('Ship') >= 1,1).otherwise(0))

travel_type3 = travel_type2.select('individual_id', 'Cruise')
travel_type3.show()

In [ ]:
from pyspark.sql.functions import when, col

# Add a cruise booking group column based on 'Ship' count
travel_type_grouped = travel_type1.withColumn(
    "cruise_booking_group",
    when(col("Ship") == 0, "None")
    .when(col("Ship") == 1, "One")
    .otherwise("More than one")
)
travel_type_grouped.groupBy("cruise_booking_group").count().orderBy("cruise_booking_group").show()


In [ ]:
data_path

In [ ]:
feature = spark.read.parquet('/net/clusterhn/home/sli/msds630/aaa_data/transformed_features_cruiser_cancellation_v2.parquet')

In [ ]:
feature1 = feature.join(travel_type3, 'individual_id')
feature1.columns

In [ ]:
feature1.write.mode("overwrite").parquet(data_path + "features_AAA_cruisebookings.parquet")

In [ ]:
## decision tree

In [ ]:
print(train_data.columns)
df = df.withColumnRenamed("Cruise", "label")


In [ ]:
# Rename 'Cruise' to 'label'
df = df.withColumnRenamed("Cruise", "label")

# Now split the data
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)

# Cache training data
train_data.cache()

print("Columns in train_data:", train_data.columns)  # Should show 'label'

# Then fit your pipeline/model
model_dt = pipeline_dt.fit(train_data)
